In [1]:
# Determining how to apply actions to Go2 joints
# Reference
# https://genesis-world.readthedocs.io/en/latest/user_guide/getting_started/control_your_robot.html#joint-control
# https://github.com/Genesis-Embodied-AI/Genesis/tree/main/examples/locomotion

# Importing libraries
import genesis as gs

#Initializes Genesis with the CPU backend.
gs.init(backend=gs.gpu)

#Create a Scene
scene = gs.Scene(show_viewer=True)

#Adds a flat ground plane to the scene.
plane = scene.add_entity(gs.morphs.Plane())

#Integrate the Go2 Robot xml.
robot = gs.morphs.MJCF(file="xml/Unitree_Go2/go2.xml")

#Add an entity to the scene.
Go2 = scene.add_entity(robot)

#Builds the scene.
scene.build()

[Genesis] [23:17:14] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [23:17:14] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [23:17:14] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [23:17:15] [INFO] Running on [NVIDIA GeForce RTX 3060 Laptop GPU] with backend gs.cuda. Device memory: 6.00 GB.
[Genesis] [23:17:15] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [23:17:16] [INFO] Scene <8096c26> created.
[Genesis] [23:17:16] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <fea5731>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [23:17:16] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <e95003b>, morph: <gs.morphs.MJCF(file='c:\Users\anich\OneDrive\Desktop\total_robotics\genesis_AI_sims\Unitree_Go2\xml\Unitree_Go2\go2.xml')>, material: <gs.materials.Rigid>.
[Genesis] [23:17:17] [WARNING] (MJCF) Friction loss at DoF-level not supported.


In [2]:
import numpy as np

# Gait frequency (Hz) and amplitude (radians)
freq = 1.0  # 1 cycle per second
amp = 0.5   # 0.5 rad swing

# Time step size (should match Genesis default)
dt = 0.01
t = 0.0

# Ordered joint list for a trot gait (same on each side, but out of phase)
joint_names = [
    'FL_hip_joint', 'FL_thigh_joint', 'FL_calf_joint',
    'FR_hip_joint', 'FR_thigh_joint', 'FR_calf_joint',
    'RL_hip_joint', 'RL_thigh_joint', 'RL_calf_joint',
    'RR_hip_joint', 'RR_thigh_joint', 'RR_calf_joint'
]

# Map joint name → dof index (you should have this already)
motor_map = {
    name: Go2.get_joint(name).dof_idx_local
    for name in joint_names
}


[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and will be removed in future release. Please use 'dof_idx_local' instead.
[Genesis] [23:19:40] [WARNING] This property is deprecated and

In [3]:
motor_map

{'FL_hip_joint': [0, 1, 2, 3, 4, 5],
 'FL_thigh_joint': 10,
 'FL_calf_joint': 14,
 'FR_hip_joint': 7,
 'FR_thigh_joint': 11,
 'FR_calf_joint': 15,
 'RL_hip_joint': 8,
 'RL_thigh_joint': 12,
 'RL_calf_joint': 16,
 'RR_hip_joint': 9,
 'RR_thigh_joint': 13,
 'RR_calf_joint': 17}

In [4]:
motor_map = {
    'FL_hip_joint': 6,   # Use the correct single DoF for actuation (NOT the first 6)
    'FL_thigh_joint': 10,
    'FL_calf_joint': 14,
    'FR_hip_joint': 7,
    'FR_thigh_joint': 11,
    'FR_calf_joint': 15,
    'RL_hip_joint': 8,
    'RL_thigh_joint': 12,
    'RL_calf_joint': 16,
    'RR_hip_joint': 9,
    'RR_thigh_joint': 13,
    'RR_calf_joint': 17,
}
motor_dofs = list(motor_map.values())  # ✅ Now 12 items, matching target_positions


In [5]:
motor_dofs

[6, 10, 14, 7, 11, 15, 8, 12, 16, 9, 13, 17]

In [6]:
Go2.set_dofs_kp(kp=np.array([3000] * len(motor_dofs)), dofs_idx_local=motor_dofs)
Go2.set_dofs_kv(kv=np.array([300] * len(motor_dofs)), dofs_idx_local=motor_dofs)


In [7]:
# Main control loop
for _ in range(1000):
    t += dt
    target_positions = []

    for joint_name in joint_names:
        phase = np.pi if ('FR' in joint_name or 'RL' in joint_name) else 0.0

        if 'hip' in joint_name:
            angle = 0.2 * np.sin(2 * np.pi * freq * t + phase)
        elif 'thigh' in joint_name:
            angle = amp * np.sin(2 * np.pi * freq * t + phase)
        else:  # calf
            angle = -amp * np.sin(2 * np.pi * freq * t + phase)

        target_positions.append(angle)

    Go2.control_dofs_position(np.array(target_positions), motor_dofs)
    scene.step()

[Genesis] [23:19:48] [INFO] Running at 0.19 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.20 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.21 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.22 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.23 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.24 FPS.
[Genesis] [23:19:48] [INFO] Running at 0.25 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.27 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.28 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.29 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.31 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.33 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.34 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.36 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.38 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.40 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.42 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.44 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.46 FPS.
[Genesis] [23:19:49] [INFO] Running at 0.49 FPS.
[Genesis] [23:19:49]

In [8]:
import numpy as np

# -------------------------------
# STEP 1: Set initial robot pose
# -------------------------------

# Set the robot's base position in the world (x, y, z)
# 0.35 meters in z means the robot starts slightly above the ground
Go2.set_pos([0, 0, 0.35])

# Set the robot's orientation using a quaternion (w, x, y, z)
# [1, 0, 0, 0] means "no rotation" — the robot stands upright
Go2.set_quat([1, 0, 0, 0])


# ---------------------------------------------------
# STEP 2: Define a stable "standing" joint configuration
# ---------------------------------------------------

# These are the target joint angles (in radians) for each leg joint to make the robot stand properly
# The thigh and calf angles make the legs slightly bent, mimicking a natural standing posture
standing_pose = {
    'FL_hip_joint': 0.0, 'FL_thigh_joint': 0.8, 'FL_calf_joint': -1.5,
    'FR_hip_joint': 0.0, 'FR_thigh_joint': 0.8, 'FR_calf_joint': -1.5,
    'RL_hip_joint': 0.0, 'RL_thigh_joint': 0.8, 'RL_calf_joint': -1.5,
    'RR_hip_joint': 0.0, 'RR_thigh_joint': 0.8, 'RR_calf_joint': -1.5,
}

# This is the order in which we will apply the joint targets to the robot
# It must match the order of indices in the `motor_dofs` list
joint_order = [
    'FL_hip_joint','FL_thigh_joint','FL_calf_joint',
    'FR_hip_joint','FR_thigh_joint','FR_calf_joint',
    'RL_hip_joint','RL_thigh_joint','RL_calf_joint',
    'RR_hip_joint','RR_thigh_joint','RR_calf_joint'
]

# Create a list of joint angles based on the standing pose and joint order
stand_targets = [standing_pose[joint] for joint in joint_order]

# Send the standing joint angles to the robot
# This tells the PD controller what angles each joint should reach
Go2.control_dofs_position(np.array(stand_targets), motor_dofs)

# Let the robot settle into this position for a short time
# 100 steps = 1 second at 0.01s per step
for _ in range(100):
    scene.step()


# -------------------------------
# STEP 3: Walking parameters
# -------------------------------

# Frequency of leg movement (1.0 = 1 complete step cycle per second)
gait_freq = 1.0

# Time step (how much time passes per simulation frame)
dt = 0.01

# Simulation time tracker
t = 0.0

# Phase offsets for each leg in the gait cycle
# FR and RL are in opposite phase from FL and RR (diagonal pairs)
phases = {
    'FL': 0.0,
    'FR': np.pi,
    'RL': np.pi,
    'RR': 0.0,
}

# List of leg names for looping
legs = ['FL', 'FR', 'RL', 'RR']

# ---------------------------------------------------
# STEP 4: Gait generator function
# ---------------------------------------------------

def compute_leg_angles(leg, t, amp=0.6):
    """
    Computes joint angles for a given leg at time t.
    amp = step height amplitude.
    Each leg follows a sine wave pattern.
    """
    phase = phases[leg]
    
    # Vertical movement (lift) via thigh and calf
    z = amp * np.sin(2 * np.pi * gait_freq * t + phase)
    
    # Small swing of the hip left/right
    hip = 0.2 * np.sin(2 * np.pi * gait_freq * t + phase)
    
    # Thigh and calf angles go up/down to mimic stepping
    thigh = 0.8 + z
    calf = -1.5 - z
    
    return [hip, thigh, calf]


# -------------------------------
# STEP 5: Walking loop
# -------------------------------

# This loop moves the legs in a walking (trot) pattern
for _ in range(1000):
    t += dt
    target_angles = []

    # Get new joint angles for each leg based on current time
    for leg in legs:
        angles = compute_leg_angles(leg, t)
        target_angles.extend(angles)  # Append 3 joint angles per leg

    # Apply the joint angles to the robot
    Go2.control_dofs_position(np.array(target_angles), motor_dofs)

    # Advance simulation
    scene.step()


[Genesis] [23:21:04] [INFO] Running at 1.75 FPS.
[Genesis] [23:21:04] [INFO] Running at 1.83 FPS.
[Genesis] [23:21:04] [INFO] Running at 1.92 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.00 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.10 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.21 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.31 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.43 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.55 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.68 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.81 FPS.
[Genesis] [23:21:04] [INFO] Running at 2.95 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.10 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.26 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.42 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.59 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.75 FPS.
[Genesis] [23:21:04] [INFO] Running at 3.94 FPS.
[Genesis] [23:21:04] [INFO] Running at 4.13 FPS.
[Genesis] [23:21:04] [INFO] Running at 4.33 FPS.
[Genesis] [23:21:04]

## RL implementation

In [9]:
import gym
from gym import spaces
import numpy as np

class UnitreeGo2Env(gym.Env):
    def __init__(self):
        super().__init__()

        # ---- ACTION SPACE ----
        # The robot has 12 joints (3 per leg).
        # Each action is a target position (in radians) for one joint.
        # Allowed range: from -1.5 to +1.5 radians
        self.action_space = spaces.Box(low=-1.5, high=1.5, shape=(12,), dtype=np.float32)

        # ---- OBSERVATION SPACE ----
        # The observation includes:
        # - Joint positions (12 values)
        # - Joint velocities (12 values)
        # Total = 24-dimensional vector
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(24,), dtype=np.float32)

        # ---- JOINT MAPPING ----
        # Each joint in the robot corresponds to a Degree of Freedom (DOF) index in Genesis
        # This map connects joint names to their local DOF index
        self.motor_map = {
            'FL_hip_joint': 6,
            'FL_thigh_joint': 10,
            'FL_calf_joint': 14,
            'FR_hip_joint': 7,
            'FR_thigh_joint': 11,
            'FR_calf_joint': 15,
            'RL_hip_joint': 8,
            'RL_thigh_joint': 12,
            'RL_calf_joint': 16,
            'RR_hip_joint': 9,
            'RR_thigh_joint': 13,
            'RR_calf_joint': 17,
        }

        # Save the DOF indices (used to control the robot)
        self.motor_dofs = list(self.motor_map.values())

        # Save the joint names (used to order the actions and observations)
        self.joint_order = list(self.motor_map.keys())

    def _get_obs(self):
        # ---- GET OBSERVATIONS FROM GENESIS ----
        # We read the current joint positions and velocities from the Genesis simulation
        qpos = Go2.get_dofs_position(self.motor_dofs).cpu().numpy()  # joint angles
        qvel = Go2.get_dofs_velocity(self.motor_dofs).cpu().numpy()  # joint angular velocities
        return np.concatenate([qpos, qvel]).astype(np.float32)  # combine into 24-length observation

    def _compute_reward(self, obs, action):
        # ---- DEFINE THE REWARD FUNCTION ----
        # Simple version: reward is how high the robot’s base is (higher = better)
        # This encourages the robot to stay upright and not fall
        base_z = Go2.get_pos()[2]  # z-position (height) of robot’s main body
        reward = base_z
        return reward

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)  # required for Gym compliance

        # ---- RESET THE ROBOT POSITION AND ORIENTATION ----
        # Start the robot just above ground, standing upright
        Go2.set_pos([0, 0, 0.35])               # Set position to (x=0, y=0, z=0.35)
        Go2.set_quat([1, 0, 0, 0])              # Set orientation: no rotation

        # ---- APPLY STANDING POSE ----
        # These angles put the robot in a stable crouching position
        standing_pose = np.array([
            0.0, 0.8, -1.5,   # FL leg: hip, thigh, calf
            0.0, 0.8, -1.5,   # FR leg
            0.0, 0.8, -1.5,   # RL leg
            0.0, 0.8, -1.5    # RR leg
        ])
        Go2.control_dofs_position(standing_pose, self.motor_dofs)

        # ---- STABILIZE ----
        # Step the simulation a bit to let robot settle
        for _ in range(100):
            scene.step()

        # Return initial observation and empty info dict
        return self._get_obs(), {}

    def step(self, action):
        # ---- APPLY ACTION TO ROBOT ----
        # Action is a 12-value array of joint target positions
        Go2.control_dofs_position(action, self.motor_dofs)

        # Step the simulation by 1 frame
        scene.step()

        # ---- COLLECT OBSERVATIONS ----
        obs = self._get_obs()

        # ---- COMPUTE REWARD ----
        reward = self._compute_reward(obs, action)

        # ---- TERMINATION CONDITION ----
        # If the robot’s base drops too low (fell down), mark episode as done
        done = Go2.get_pos()[2] < 0.2

        truncated = False  # Not using truncation yet

        # Return everything needed by the PPO agent
        return obs, reward, done, truncated, {}

    def render(self):
        # Genesis has a built-in viewer already
        pass


In [10]:
# Import the custom Gym environment we created earlier for the Unitree Go2 robot
# This class handles simulation reset, step logic, observations, actions, and rewards


# from unitree_env import UnitreeGo2Env

# Create an instance of the environment
# This will:
# - Load the robot in Genesis World
# - Set up action and observation spaces
# - Prepare the simulation to start training or testing
env = UnitreeGo2Env()


In [11]:
from stable_baselines3 import PPO
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100_000)
model.save("ppo_unitree_go2")


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\anich\.conda\envs\total_robotics\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


[Genesis] [23:22:58] [INFO] Running at 0.43 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.45 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.48 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.50 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.53 FPS.


c:\Users\anich\.conda\envs\total_robotics\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


[Genesis] [23:22:58] [INFO] Running at 0.56 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.59 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.62 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.65 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.68 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.72 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.75 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.79 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.83 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.87 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.91 FPS.
[Genesis] [23:22:58] [INFO] Running at 0.96 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.01 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.06 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.11 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.17 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.23 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.29 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.35 FPS.
[Genesis] [23:22:58] [INFO] Running at 1.42 FPS.
[Genesis] [23:22:59]

KeyboardInterrupt: 

In [12]:
obs, _ = env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, _ = env.step(action)
    if done or truncated:
        obs, _ = env.reset()


[Genesis] [23:25:09] [INFO] Running at 0.37 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.39 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.41 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.43 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.46 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.48 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.50 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.53 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.55 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.58 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.61 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.64 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.67 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.71 FPS.
[Genesis] [23:25:10] [INFO] Running at 0.74 FPS.
[Genesis] [23:25:11] [INFO] Running at 0.78 FPS.
[Genesis] [23:25:11] [INFO] Running at 0.82 FPS.
[Genesis] [23:25:11] [INFO] Running at 0.86 FPS.
[Genesis] [23:25:11] [INFO] Running at 0.90 FPS.
[Genesis] [23:25:11] [INFO] Running at 0.94 FPS.
[Genesis] [23:25:11]

In [ ]:
from stable_baselines3 import PPO
import numpy as np

# Load your environment and trained model
env = UnitreeGo2Env()
model = PPO.load("ppo_unitree_go2")

# Reset environment and get initial observation
obs, _ = env.reset()

# Run the trained policy in a loop
for _ in range(1000):  # or while True
    # Predict the next action using the trained policy
    action, _ = model.predict(obs, deterministic=True)

    # Step the environment (and underlying Genesis simulation)
    obs, reward, done, truncated, info = env.step(action)

    # Optional: render if you implemented it
    env.render()

    # Break loop if episode ends
    if done or truncated:
        obs, _ = env.reset()
